# VarNet brain baseline (U5 only)

Runs the pretrained brain VarNet checkpoint (E2E, 12 cascades) on the same
12-volume / 192-slice brain validation set used for the diffusion sweeps.
Inference only - no training.

**Prereqs in `MyDrive/fastmri_artifacts/`:** `brain_12vols.tar`,
`brain_leaderboard_state_dict.pt` (the 114 MB VarNet checkpoint - upload it
from your local `checkpoints/varnet/`).

**Runtime -> T4 GPU.**


In [ ]:
# Cell 1 - verify GPU
!nvidia-smi | head -20


In [ ]:
# Cell 2 - clone repo + ADPS dnnlib/torch_utils (needed by EDM pickle).
import os
%cd /content
if os.path.isdir('/content/fastmri/.git'):
    %cd /content/fastmri
    !git fetch --quiet && git reset --hard origin/main
else:
    !rm -rf fastmri
    !git clone https://github.com/carlo-scr/fastmri.git
    %cd /content/fastmri
if not os.path.isdir('external/adps/dnnlib'):
    !rm -rf external/adps
    !git clone --depth 1 https://github.com/utcsilab/ambient-diffusion-mri.git external/adps
!git --no-pager log -1 --oneline


In [ ]:
# Cell 3 - install deps (do NOT pin numpy; Colab's scikit-image needs >=2.3).
!pip install -q h5py s3fs wandb pyyaml fastmri


In [ ]:
# Cell 4 - mount Drive, stage brain data + VarNet checkpoint.
# Expects in MyDrive/fastmri_artifacts/:
#   - brain_12vols.tar             (brain val data)
#   - brain_leaderboard_state_dict.pt  (VarNet brain checkpoint, 114 MB)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
ART = '/content/drive/MyDrive/fastmri_artifacts'
assert os.path.exists(f'{ART}/brain_12vols.tar'), 'brain tarball missing in Drive'
assert os.path.exists(f'{ART}/brain_leaderboard_state_dict.pt'), \
    'VarNet checkpoint missing in Drive (brain_leaderboard_state_dict.pt)'

os.makedirs('checkpoints/varnet', exist_ok=True)
shutil.copy(f'{ART}/brain_leaderboard_state_dict.pt',
            'checkpoints/varnet/brain_leaderboard_state_dict.pt')

!tar xf "$ART/brain_12vols.tar" -C /content/fastmri
!find data/multicoil_val -name '._*' -delete
!find data/multicoil_val -name '.DS_Store' -delete

import h5py, glob
files = sorted(glob.glob('data/multicoil_val/*.h5'))
bad = []
for f in files:
    try:
        with h5py.File(f, 'r'): pass
    except Exception as e:
        bad.append((f, str(e)))
print(f'{len(files)} h5 files; {len(bad)} unreadable')
!ls -la checkpoints/varnet/


In [ ]:
# Cell U5 - VarNet brain re-eval over 12 validation volumes (inference only).
# IMPORTANT: chdir OUT of /content/fastmri before import so the pip-installed
# 'fastmri' package wins over the repo dir (which is not a Python package).
import os, sys, glob, json
REPO = '/content/fastmri'
os.chdir('/content')
sys.path = [p for p in sys.path if p not in (REPO, '')]
sys.path.append(REPO)   # for scripts.run_varnet_baseline

import numpy as np, h5py, torch
from fastmri.models import VarNet           # pip-installed pkg
from scripts.run_varnet_baseline import run_varnet_on_brain, create_equispaced_mask

model = VarNet(num_cascades=12, pools=4, chans=18, sens_pools=4, sens_chans=8)
sd = torch.load(f'{REPO}/checkpoints/varnet/brain_leaderboard_state_dict.pt',
                map_location='cpu', weights_only=False)
model.load_state_dict(sd); model.eval()

VOLS = sorted(glob.glob(f'{REPO}/data/multicoil_val/file_brain_AXT2_*.h5'))[:12]
SLICES_PER_VOL = list(range(0, 16))   # 12 x 16 = 192

all_results = {'R4': {}, 'R8': {}}
for R in (4, 8):
    print(f'\n=== VarNet brain R={R} ===')
    for vp in VOLS:
        print(' vol:', os.path.basename(vp))
        all_results[f'R{R}'][os.path.basename(vp)] = run_varnet_on_brain(
            model, vp, SLICES_PER_VOL, acceleration=R)

os.makedirs(f'{REPO}/outputs', exist_ok=True)
OUT = f'{REPO}/outputs/varnet_baseline_12vol.json'
with open(OUT, 'w') as f:
    json.dump(all_results, f, indent=2)
print('saved ->', OUT)

for R in (4, 8):
    psnrs = [s['psnr'] for v in all_results[f'R{R}'].values() for s in v]
    ssims = [s['ssim'] for v in all_results[f'R{R}'].values() for s in v]
    print(f' R={R}: n={len(psnrs)} PSNR={np.mean(psnrs):.2f}+-{np.std(psnrs):.2f} '
          f'SSIM={np.mean(ssims):.4f}+-{np.std(ssims):.4f}')


In [ ]:
# Cell - back up VarNet results to Drive.
import shutil
ART = '/content/drive/MyDrive/fastmri_artifacts'
src = '/content/fastmri/outputs/varnet_baseline_12vol.json'
shutil.copy(src, f'{ART}/varnet_baseline_12vol.json')
print('Saved:', f'{ART}/varnet_baseline_12vol.json')
